[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/sif_feasibility_test.ipynb)

# Can an Ersilia `.sif` run in Google Colab?

Sandbox feasibility test. **Not** workshop material: it lives in `sandbox/`, is not linked
from any project README, and is not covered by the notebook rules in `CLAUDE.md`.

Run top to bottom on a **CPU** runtime. Every step prints `PASS` or `FAIL` on its own line,
so the whole output can be pasted back verbatim.

**Background.** Colab deliberately blocked the `/proc` bind mounts Apptainer needs in March 2025
([colabtools#5173](https://github.com/googlecolab/colabtools/issues/5173)); it was reported
working again in November 2025 via an `unshare -r` wrapper.

## Answer: yes, with three corrections

Confirmed end to end on 25 September 2026 (Ubuntu 24.04.5, kernel 6.6.122+, Apptainer 1.5.4,
2 CPUs / 12 GB RAM). Three assumptions this test started from turned out to be wrong, and each
is now a section below:

1. **Plain `apptainer` still fails**; every call needs the `unshare -r` wrapper (section 3).
2. **`run <sif> input.csv output.csv` does not work.** The image's entrypoint ignores its
   arguments and starts a web server, and it looks for its bundle in the wrong place
   (section 5). The image is driven by serving it and POSTing to it (sections 6-7).
3. **`--no-home` is the wrong flag here.** It is what forces `--writable-tmpfs`; without it the
   container writes to Colab's own writable `/root` and no extra flag is needed (section 6).

**The image under test:** `https://models-sif.s3.eu-north-1.amazonaws.com/eos42ez_v1.sif`
(public, 4.54 GB, built 8 September 2026 from `docker://ersiliaos/eos42ez:v1.1.0`).
`eos42ez` is *antibiotics-ai-cytotox*, which predicts cytotoxicity against three cell lines.


## 1. Environment

Records the Ubuntu and kernel version, and whether unprivileged user namespaces exist at all.
`max_user_namespaces: 0` means the `unshare -r` recipe is dead on arrival - stop there.


In [ ]:
%%bash
echo "=== STEP 1: environment ==="
head -2 /etc/os-release
echo "kernel: $(uname -r)"
echo "whoami: $(id -un) ($(id -u))   HOME=$HOME"
echo "max_user_namespaces: $(cat /proc/sys/user/max_user_namespaces 2>/dev/null || echo UNREADABLE)"
echo "cpus: $(nproc)   ram: $(free -g | awk '/^Mem:/{print $2}')GB   free disk: $(df -h /content | awk 'NR==2{print $4}')"


## 2. Install Apptainer

The plain `apptainer` package, not `apptainer-suid` - the user-namespace path is the one that
works in a sandbox. Falls back to the upstream `.deb` if the PPA has no build for Colab's
Ubuntu release.


In [ ]:
%%bash
echo "=== STEP 2: install apptainer ==="
if ! command -v apptainer >/dev/null 2>&1; then
  (add-apt-repository -y ppa:apptainer/ppa && apt-get update -qq && apt-get install -y apptainer) >/tmp/apt.log 2>&1
fi
if ! command -v apptainer >/dev/null 2>&1; then
  echo "PPA route failed; falling back to the upstream .deb"
  VER=$(curl -s https://api.github.com/repos/apptainer/apptainer/releases/latest | sed -n 's/.*"tag_name": *"v\([^"]*\)".*/\1/p')
  echo "latest release: ${VER}"
  curl -sLO "https://github.com/apptainer/apptainer/releases/download/v${VER}/apptainer_${VER}_amd64.deb"
  apt-get install -y "./apptainer_${VER}_amd64.deb" >>/tmp/apt.log 2>&1
fi
if command -v apptainer >/dev/null 2>&1; then
  apptainer --version
  echo PASS
else
  echo "FAIL - no apptainer binary; tail of the log:"
  tail -20 /tmp/apt.log
fi


## 3. Does any container run at all?

The decision point, reached before spending 4.5 GB of download. Tries the call unwrapped and
then wrapped in `unshare -r`. If **neither** marker prints, containers are still blocked and
the answer is no.

The unwrapped call is expected to fail. What it prints is the March 2025 block, still in force:

```
FATAL: ... while mounting overlay: can't mount overlay filesystem to
/var/lib/apptainer/mnt/session/final: while setting effective capabilities:
CAP_DAC_READ_SEARCH is not in the permitted capability set
```

`unshare -r` creates a user namespace in which the process holds that capability, and the same
call then succeeds. Everything below therefore goes through `unshare -r`.


In [ ]:
%%bash
echo "=== STEP 3: smoke test ==="
cd /content
rm -f alpine.sif
apptainer pull alpine.sif docker://alpine:latest >/tmp/pull.log 2>&1 \
  || unshare -r apptainer pull alpine.sif docker://alpine:latest >/tmp/pull.log 2>&1
if [ ! -f alpine.sif ]; then
  echo "FAIL - could not even build a SIF; tail of the log:"; tail -20 /tmp/pull.log; exit 0
fi
echo "--- plain ---"
apptainer exec /content/alpine.sif echo PLAIN_OK 2>&1 | tail -5
echo "--- unshare -r ---"
unshare -r apptainer exec /content/alpine.sif echo UNSHARE_OK 2>&1 | tail -5
echo "(PASS if UNSHARE_OK appeared; PLAIN_OK is not expected)"


## 4. Download the model image

The bucket is public, so this is a plain HTTPS download - no credentials. 4.54 GB, re-fetched on
every fresh runtime; `-c` lets it resume if the connection drops.

Measured at **25.4 MB/s**, so about **2.8 minutes**. That is the number that decides whether this
could ever be workshop material - see section 8.


In [ ]:
%%bash
echo "=== STEP 4: download the SIF ==="
cd /content
URL="https://models-sif.s3.eu-north-1.amazonaws.com/eos42ez_v1.sif"
EXPECTED=4541046784
if [ -f eos42ez_v1.sif ] && [ "$(stat -c %s eos42ez_v1.sif)" = "$EXPECTED" ]; then
  echo "already present and complete, skipping download"
else
  time wget -c -q --show-progress "$URL" -O eos42ez_v1.sif
fi
ls -lh eos42ez_v1.sif
ACTUAL=$(stat -c %s eos42ez_v1.sif)
if [ "$ACTUAL" = "$EXPECTED" ]; then
  echo "PASS (size matches)"
else
  echo "FAIL - got $ACTUAL bytes, expected $EXPECTED"
fi


## 5. What is actually inside

This is where the original plan for this notebook broke, so it is worth printing in full.

The image's runscript is `sh docker-entrypoint.sh`, and that script is four lines long: it
checks `$MODEL` is set and then runs
`ersilia_model_serve --bundle_path /root/bundles/$MODEL --port 80`. Two consequences:

- **It ignores its arguments.** Passing `input.csv output.csv` to `apptainer run` does nothing;
  the container starts a FastAPI server instead of writing a file.
- **The path is wrong for this image.** The bundle ships at `/opt/ersilia/bundles/eos42ez`
  (`ERSILIA_PATH=/opt/ersilia`), not `/root/bundles/eos42ez`, so the entrypoint dies with
  `FileNotFoundError` even when it is reached. Reaching it at all also needs `--pwd /root`,
  because the runscript calls `docker-entrypoint.sh` by a relative path.

So the entrypoint is unusable as shipped, and section 6 bypasses it.

The `%environment` block pins `HOME=/root` only when `[ -d /root/.lazyqsar ]`. **This image has
no such cache**, so that pin never fires and `--no-home` has nothing to protect against here.

> **Note:** these three checks pass `--no-home`, and section 6 deliberately does not. The flag
> decides *whose* `/root` you see: with it, the container's (what we want to inspect here);
> without it, Colab's (what we want to write to there). Inspecting the image without
> `--no-home` silently reports the host and makes the entrypoint look fine.


In [ ]:
%%bash
echo "=== STEP 5: container interface ==="
SIF=/content/eos42ez_v1.sif
# --no-home here ONLY: it exposes the container's own /root. Without it Colab's /root is
# bind-mounted on top and these checks would describe the host instead of the image.
echo "--- the entrypoint it would run ---"
unshare -r apptainer exec --no-home "$SIF" cat /root/docker-entrypoint.sh 2>&1
echo "--- where the bundle really is ---"
unshare -r apptainer exec --no-home "$SIF" sh -c 'echo "in the image: $(ls /opt/ersilia/bundles)"; ls -d /root/bundles 2>/dev/null || echo "/root/bundles: absent, so the entrypoint would fail here"' 2>&1
echo "--- lazyqsar cache? ---"
unshare -r apptainer exec --no-home "$SIF" sh -c 'ls -d /root/.lazyqsar 2>/dev/null || echo "no lazyqsar cache, so the HOME=/root pin never fires"' 2>&1


## 6. Serve the model

Bypasses the broken entrypoint and calls `ersilia_model_serve` directly with the bundle path
that exists. The server is started detached so the notebook can carry on.

Two flags deliberately **not** used:

- **`--no-home`** - it hides Colab's `/root`, and the app creates `/root/eos` at import time.
  Inside the read-only squashfs that raises `OSError: [Errno 30] Read-only file system`.
- **`--writable-tmpfs`** - only needed to repair the damage `--no-home` causes. Left out, the
  container writes `/root/eos` to Colab's own disk, which is writable and not size-capped.

Apptainer shares the host network, so the server is reachable on `127.0.0.1:8000`.

> **Note:** the `unshare -r apptainer exec` wrapper exits as soon as it has spawned the server,
> so the process actually holding the port is a `run_uvicorn.py` child. A `pkill -f
> ersilia_model_serve` matches nothing, the relaunch then fails with `CalledProcessError`
> (address in use), and a naive health check happily passes against the **old** server. The
> cell below kills both patterns and refuses to launch until the port is free.


In [ ]:
%%bash
echo "=== STEP 6: serve the model ==="
# The wrapper exits immediately; the real server is a run_uvicorn.py child, so match both.
# Matching only ersilia_model_serve kills nothing and leaves a stale server on the port.
pkill -f 'ersilia_model_serve|run_uvicorn' 2>/dev/null
for i in $(seq 30); do
  ss -lntH 'sport = :8000' | grep -q . || break
  sleep 1
done
if ss -lntH 'sport = :8000' | grep -q .; then
  echo "FAIL - port 8000 still held after 30s"; exit 1
fi
echo "port 8000 free"
rm -f /content/serve.log
nohup unshare -r apptainer exec --bind /content:/content /content/eos42ez_v1.sif \
  ersilia_model_serve --bundle_path /opt/ersilia/bundles/eos42ez --port 8000 \
  > /content/serve.log 2>&1 &
echo "launched detached, pid $!"


Waits for the server to answer `/healthz` rather than sleeping a fixed number of seconds.
Startup took about 20 seconds on a 2-CPU runtime.


In [ ]:
import pathlib
import time
import urllib.error
import urllib.request

log = pathlib.Path("/content/serve.log")
start = time.time()
deadline = start + 180
while time.time() < deadline:
    if "CalledProcessError" in log.read_text(errors="ignore"):
        print("FAIL - the server process died; tail of /content/serve.log:")
        print("\n".join(log.read_text(errors="ignore").splitlines()[-5:]))
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/healthz", timeout=5) as response:
            if response.status == 200:
                print(f"PASS - server up after {time.time() - start:.0f}s")
                break
    except (urllib.error.URLError, OSError):
        time.sleep(3)
else:
    print("FAIL - server never came up; check /content/serve.log")


## 7. Predict

Three familiar molecules. `POST /run` takes a plain JSON list of SMILES and returns one object
per molecule, so the result drops straight into a dataframe.

Ethanol, aspirin and caffeine should all score **low**, which is the sanity check that the model
is really running and not returning a constant.


In [ ]:
import json
import time
import urllib.request

import pandas as pd

smiles = [
    "CCO",                           # ethanol
    "CC(=O)Oc1ccccc1C(=O)O",         # aspirin
    "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",  # caffeine
]
request = urllib.request.Request(
    "http://127.0.0.1:8000/run",
    data=json.dumps(smiles).encode(),
    headers={"Content-Type": "application/json"},
)
start = time.time()
with urllib.request.urlopen(request, timeout=300) as response:
    predictions = json.loads(response.read())
print(f"inference wall clock: {time.time() - start:.1f}s")

df = pd.DataFrame(predictions, index=["ethanol", "aspirin", "caffeine"])
df


In [ ]:
ok = len(df) == len(smiles) and df.notna().all().all()
print("VERDICT: an Ersilia SIF DOES run in Google Colab" if ok
      else "VERDICT: no usable output - see the failures above")


## 8. What this means

**It works.** The recipe, in one line:

```bash
unshare -r apptainer exec --bind /content:/content <sif> \
  ersilia_model_serve --bundle_path /opt/ersilia/bundles/<model> --port 8000
```

then `POST http://127.0.0.1:8000/run` with a JSON list of SMILES.

**The cost, measured on a free CPU runtime:**

| Step | Cost |
|---|---|
| Apptainer install | ~40 s |
| SIF download | 4.54 GB at 25.4 MB/s, about 2.8 min, on every fresh runtime |
| Server startup | ~20 s |
| Inference, 3 molecules, cold | 45 s |

**Is this workshop material?** Not as it stands. Roughly four minutes of setup before the first
prediction, repeated every time a runtime is recycled, is a lot to ask of participants - and the
25 MB/s measured here is Colab's link to S3, not anyone's link in Buea. It is also 4.54 GB per
participant per session against the bucket.

Better uses of this result:

- Keep it as the reference recipe for when a model has **no** other route into Colab.
- For the blue group specifically, `eos42ez` is the cytotoxicity model behind
  `blue_cytotoxicity_filter.ipynb`. If that notebook ever needs to run the model itself rather
  than read precomputed values, this is how - but precomputing the predictions once and
  committing the CSV is far cheaper for participants.

**Two bugs worth reporting upstream** (both in the image, not in Colab):

1. `docker-entrypoint.sh` hardcodes `/root/bundles/$MODEL`, but the bundle ships at
   `$ERSILIA_PATH/bundles/$MODEL`. The entrypoint cannot work for this image.
2. The runscript invokes `docker-entrypoint.sh` by a relative path, so `apptainer run` needs
   `--pwd /root` even once the path above is fixed.
